# Análisis Enriquecido de Charts Musicales

**Data Source:** YouTube Charts enriched with country, genre, and video metrics
**Week:** 2026-W13
**Generated:** 2026-03-30 06:26:57
**AI Analysis:** Powered by DeepSeek API


## 1. Configuración y Carga de Datos

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import sqlite3
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib for inline display in notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Reds")

YT_RED = '#FF0000'
YT_RED_DARK = '#CC0000'
YT_BG = '#FFFFFF'
YT_SURFACE = '#F9F9F9'
YT_TEXT = '#0F0F0F'
YT_GRAY = '#606060'
YT_GRID = '#E5E5E5'

def format_number(x):
    if pd.isna(x): return x
    if x >= 1_000_000_000: return f"{x/1_000_000_000:.1f}B"
    if x >= 1_000_000: return f"{x/1_000_000:.1f}M"
    if x >= 1_000: return f"{x/1_000:.1f}K"
    return f"{x:,.0f}"

# Load data
print(f"Loading data from: youtube_charts_2026-W13_enriched.db")
conn = sqlite3.connect("youtube_charts_2026-W13_enriched.db")

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(f"Tables found: {[t[0] for t in tables]}")

if not tables:
    raise ValueError(f"No tables found in database: youtube_charts_2026-W13_enriched.db")

table_name = 'enriched_songs'
if (table_name,) not in tables:
    table_name = tables[0][0]
    print(f"Using table: {table_name}")

df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
conn.close()

df['upload_date'] = pd.to_datetime(df['upload_date'], errors='coerce')
df['upload_quarter'] = df['upload_date'].dt.quarter
df['engagement'] = (df['likes'] / df['views'] * 100).round(2)

print(f"Loaded {len(df)} songs, {df.shape[1]} columns")
df.head()


## 2. Vista Previa de los Datos

In [ ]:
df.head()

## 3. Estadísticas Generales

In [ ]:

stats = pd.DataFrame({
    'Total Songs': [100],
    'Unique Countries': [30],
    'Unique Genres': [21],
    'Total Views': [1151575574],
    'Total Likes': [133935078],
    'Total Comments': [4388628],
    'Avg Views': [11515756],
    'Avg Likes': [1339351]
})

print("GENERAL STATISTICS")
display(stats)



### 🤖 Análisis de IA

**Análisis de Estadísticas Musicales**

La colección muestra una notable diversidad, con canciones de 30 países y 21 géneros, indicando un alcance cultural amplio. Los altos promedios de vistas (11.5M) y likes (1.3M) reflejan un fuerte compromiso del público, con una tasa de likes/vistas del ~11.6%, que sugiere una recepción positiva consistente. En conjunto, se trata de un catálogo globalmente diverso y altamente atractivo, con un rendimiento de engagement sólido y uniforme.


## 4. Análisis por País

### 4.1. Distribución por Continente

In [ ]:

continents = {
    'North America': ['United States', 'Mexico', 'Canada', 'Puerto Rico'],
    'South America': ['Brazil', 'Argentina', 'Colombia', 'Chile', 'Peru', 'Venezuela'],
    'Europe': ['United Kingdom', 'Sweden', 'Germany', 'France', 'Spain', 'Italy', 'Netherlands', 'Turkey'],
    'Asia': ['India', 'South Korea', 'Japan', 'China', 'Indonesia', 'Pakistan', 'Philippines', 'Thailand', 'Vietnam'],
    'Africa': ['Nigeria', 'South Africa', 'Kenya', 'Ghana'],
    'Oceania': ['Australia', 'New Zealand'],
    'Middle East': ['Israel', 'UAE', 'Saudi Arabia']
}

def get_continent(country):
    for continent, countries in continents.items():
        if country in countries:
            return continent
    return 'Other'

df['continent'] = df['artist_country'].apply(get_continent)

continent_stats = df.groupby('continent').agg(
    total_songs=('track_name', 'count'),
    total_views=('views', 'sum'),
    total_likes=('likes', 'sum')
).reset_index().sort_values('total_songs', ascending=False)

print("\nCONTINENT STATISTICS:")
display(continent_stats)

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(YT_BG)
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(continent_stats)))

wedges, texts, autotexts = ax.pie(
    continent_stats['total_songs'],
    labels=continent_stats['continent'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(10)

ax.set_title('Song Distribution by Continent', fontweight='bold', color=YT_TEXT, fontsize=14, pad=20)
plt.tight_layout()
plt.show()


### 4.2. Top Países por Cantidad de Canciones

In [ ]:

top_countries = (df
    .groupby('artist_country')
    .agg(total_songs=('rank', 'count'), total_views=('views', 'sum'))
    .reset_index()
    .sort_values('total_songs', ascending=False)
    .head(10))

total = top_countries['total_songs'].sum()
top_countries['percentage'] = (top_countries['total_songs'] / total * 100).round(2)

print("\nTOP 10 COUNTRIES BY SONG COUNT")
display(top_countries)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(YT_BG)
ax.set_facecolor(YT_SURFACE)

colors = plt.cm.Reds(np.linspace(0.4, 1, len(top_countries)))[::-1]

bars = ax.barh(top_countries['artist_country'], top_countries['total_songs'],
               color=colors, edgecolor='#CC0000', height=0.65, alpha=0.9)

ax.set_xlabel('Number of Songs', fontsize=11, color=YT_GRAY)
ax.set_title('Top 10 Countries by Song Count', fontweight='bold', color=YT_TEXT, fontsize=14)
ax.invert_yaxis()
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_color(YT_GRID)
ax.xaxis.grid(True, color=YT_GRID, linestyle='--', alpha=0.7)

for bar, val in zip(bars, top_countries['total_songs']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{int(val)}',
            va='center', fontsize=10, fontweight='bold', color=YT_TEXT)

plt.tight_layout()
plt.show()



### 🤖 Análisis de IA

**Análisis de distribución geográfica de canciones:**

India y Estados Unidos dominan claramente el ranking, representando más de la mitad del total de canciones. Se observa una fuerte diversidad geográfica, con presencia destacada de Asia (India, Corea del Sur, Pakistán), América (EE.UU., México, Brasil, Puerto Rico) y Europa (Reino Unido). Esta distribución probablemente refleja tanto el tamaño de sus industrias musicales como la creciente globalización y popularidad de géneros como el K-pop y la música latina en listados internacionales.


### 4.3. Top Países por Total de Likes

In [ ]:

top_likes = (df
    .groupby('artist_country')['likes']
    .sum()
    .reset_index()
    .rename(columns={'likes': 'total_likes'})
    .sort_values('total_likes', ascending=False)
    .head(10))

def format_likes(x):
    if x >= 1_000_000: return f"{x/1_000_000:.1f}M"
    if x >= 1_000: return f"{x/1_000:.1f}K"
    return str(x)

top_likes['total_likes_fmt'] = top_likes['total_likes'].apply(format_likes)

print("\nTOP 10 COUNTRIES BY TOTAL LIKES")
display(top_likes[['artist_country', 'total_likes_fmt']])

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(YT_BG)
ax.set_facecolor(YT_SURFACE)

colors = plt.cm.Reds(np.linspace(0.4, 1, len(top_likes)))[::-1]

bars = ax.barh(top_likes['artist_country'], top_likes['total_likes'],
               color=colors, edgecolor='#CC0000', height=0.65, alpha=0.9)

ax.set_xlabel('Total Likes', fontsize=11, color=YT_GRAY)
ax.set_title('Top 10 Countries by Total Likes', fontweight='bold', color=YT_TEXT, fontsize=14)
ax.invert_yaxis()
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_color(YT_GRID)
ax.xaxis.grid(True, color=YT_GRID, linestyle='--', alpha=0.7)

for bar, val in zip(bars, top_likes['total_likes']):
    ax.text(val + 0.5e6, bar.get_y() + bar.get_height()/2,
            format_likes(val), va='center', fontsize=10, fontweight='bold', color=YT_TEXT)

plt.tight_layout()
plt.show()



### 🤖 Análisis de IA

Estados Unidos e India lideran significativamente en compromiso, generando más del 50% de los "likes" totales. La lista por "likes" difiere de un posible top por canciones, ya que incluye países con audiencias digitales muy activas pero mercados musicales más pequeños, como Dinamarca y Suecia. Esto sugiere que el compromiso no solo refleja el volumen de música, sino también la intensidad de la base de seguidores y la cultura digital local.


### 4.4. Top 5 Canciones por País

In [ ]:

print("\n" + "="*80)
print("TOP 5 SONGS BY COUNTRY (Views & Likes)")
print("="*80)

top_countries_list = df['artist_country'].value_counts().head(10).index.tolist()

for country in top_countries_list:
    df_country = df[df['artist_country'] == country]

    print(f"\n{country}:")

    top_views = df_country.nlargest(5, 'views')[['track_name', 'artist_names', 'views', 'likes', 'engagement']].copy()
    top_views['views'] = top_views['views'].apply(format_number)
    top_views['likes'] = top_views['likes'].apply(format_number)

    print("   Top 5 by views:")
    for _, row in top_views.iterrows():
        print(f"      - {row['track_name']} - {row['artist_names']}: {row['views']} views | {row['likes']} likes | {row['engagement']:.1f}% engagement")

    top_likes_country = df_country.nlargest(5, 'likes')[['track_name', 'artist_names', 'views', 'likes', 'engagement']].copy()
    top_likes_country['views'] = top_likes_country['views'].apply(format_number)
    top_likes_country['likes'] = top_likes_country['likes'].apply(format_number)

    print("   Top 5 by likes:")
    for _, row in top_likes_country.iterrows():
        print(f"      - {row['track_name']} - {row['artist_names']}: {row['likes']} likes | {row['views']} views | {row['engagement']:.1f}% engagement")


## 5. Análisis por Género

In [ ]:

genre_stats = (df
    .groupby('macro_genre')
    .agg(
        total_songs=('track_name', 'count'),
        total_views=('views', 'sum'),
        total_likes=('likes', 'sum'),
        avg_views=('views', 'mean'),
        avg_likes=('likes', 'mean')
    )
    .reset_index()
    .sort_values('total_songs', ascending=False))

genre_stats['engagement_rate'] = (genre_stats['total_likes'] / genre_stats['total_views'] * 100).round(2)
genre_stats['engagement_rate'] = genre_stats['engagement_rate'].fillna(0)

print("\nTOP 10 GENRES")
display(genre_stats.head(10)[['macro_genre', 'total_songs', 'engagement_rate']])


### 5.1. Treemap de Distribución de Géneros

In [ ]:

fig = px.treemap(
    genre_stats,
    path=['macro_genre'],
    values='total_songs',
    color='total_songs',
    color_continuous_scale='Reds',
    title='Genre Distribution by Song Count',
    hover_data={'engagement_rate': ':.2f', 'total_views': ':,.0f', 'total_likes': ':,.0f'}
)
fig.update_traces(textinfo="label+value", textfont_size=12)
fig.update_layout(width=1000, height=600, paper_bgcolor='white')
fig.show()


### 5.2. Tasa de Engagement por Género

In [ ]:

print("="*80)
print("ENGAGEMENT ANALYSIS BY GENRE")
print("="*80)

engagement_chart = genre_stats.sort_values('engagement_rate', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(YT_BG)
ax.set_facecolor(YT_SURFACE)

colors = plt.cm.Reds(np.linspace(0.4, 1, len(engagement_chart)))[::-1]

bars = ax.barh(engagement_chart['macro_genre'], engagement_chart['engagement_rate'],
               color=colors, edgecolor='#CC0000', height=0.65, alpha=0.9)

ax.set_xlabel('Engagement Rate (%)', fontsize=11, color=YT_GRAY)
ax.set_title('Top 10 Genres by Engagement Rate (Likes/Views)',
             fontweight='bold', color=YT_TEXT, fontsize=14)
ax.invert_yaxis()
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_color(YT_GRID)
ax.xaxis.grid(True, color=YT_GRID, linestyle='--', alpha=0.7)

for bar, val in zip(bars, engagement_chart['engagement_rate']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=10, fontweight='bold', color=YT_TEXT)

avg_engagement = genre_stats['engagement_rate'].mean()
ax.axvline(x=avg_engagement, color=YT_RED, linestyle='--', linewidth=2, alpha=0.9)
ax.text(avg_engagement + 0.1, len(engagement_chart) - 0.5,
        f'Average: {avg_engagement:.1f}%',
        fontsize=9, color=YT_RED, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nENGAGEMENT STATISTICS")
print(f"   Average: {avg_engagement:.2f}%")
print(f"   Median: {genre_stats['engagement_rate'].median():.2f}%")
print(f"   Max: {genre_stats['engagement_rate'].max():.2f}% ({genre_stats.loc[genre_stats['engagement_rate'].idxmax(), 'macro_genre']})")
print(f"   Min: {genre_stats['engagement_rate'].min():.2f}% ({genre_stats.loc[genre_stats['engagement_rate'].idxmin(), 'macro_genre']})")



### 🤖 Análisis de IA

Los géneros con mayor y menor engagement son R&B/Soul (160.4%) y K-Pop/K-Rock (13.5%), respectivamente. Esta brecha podría deberse a la profundidad emocional y nicho fiel del R&B, frente a la saturación y audiencia más amplia pero menos comprometida del K-Pop. Para creadores de contenido, esto sugiere enfocarse en géneros con comunidades dedicadas o explorar fusiones innovadoras que capturen la autenticidad del R&B con el alcance masivo de géneros pop.


### 5.3. Heatmap País-Género

In [ ]:

df_analysis = df[~df['artist_country'].isin(['Multi-country', 'Unknown'])]

matrix = pd.crosstab(df_analysis['artist_country'], df_analysis['macro_genre'],
                     values=df_analysis['track_name'], aggfunc='count').fillna(0)

top_countries_matrix = matrix.sum(axis=1).sort_values(ascending=False).head(12).index
top_genres_matrix = genre_stats.nlargest(10, 'total_songs')['macro_genre'].tolist()
top_genres_existing = [g for g in top_genres_matrix if g in matrix.columns]

matrix_heatmap = matrix.loc[top_countries_matrix, top_genres_existing]

print("="*80)
print("COUNTRY vs GENRE MATRIX (Top 12 countries × Top 10 genres)")
print("="*80)
display(matrix_heatmap)

fig = go.Figure(data=go.Heatmap(
    z=matrix_heatmap.values,
    x=matrix_heatmap.columns.tolist(),
    y=matrix_heatmap.index.tolist(),
    colorscale='Reds',
    text=matrix_heatmap.values,
    texttemplate='%{text}',
    textfont={"size": 10},
    hoverongaps=False,
    hovertemplate='<b>Country: %{y}</b><br><b>Genre: %{x}</b><br><b>Songs: %{z}</b><extra></extra>'
))

fig.update_layout(
    title='Country vs Genre Distribution',
    title_font_size=18,
    xaxis_title='Music Genre',
    yaxis_title='Country',
    xaxis_tickangle=-45,
    width=1200,
    height=700,
    paper_bgcolor='white'
)
fig.show()


## 6. Métricas de Canciones

### 6.1. Top Canciones por Vistas

In [ ]:

print("="*80)
print("TOP 10 SONGS BY VIEWS")
print("="*80)
display(df.nlargest(10, 'views')[['rank', 'track_name', 'artist_names', 'views', 'artist_country']])


### 6.2. Top Canciones por Likes

In [ ]:

print("="*80)
print("TOP 10 SONGS BY LIKES")
print("="*80)
display(df.nlargest(10, 'likes')[['rank', 'track_name', 'artist_names', 'likes', 'artist_country']])


### 6.3. Top Canciones por Engagement

In [ ]:

print("="*80)
print("TOP 10 SONGS BY ENGAGEMENT (Likes/Views %)")
print("="*80)
display(df.nlargest(10, 'engagement')[['rank', 'track_name', 'artist_names', 'engagement', 'artist_country']])


## 7. Métricas de Video

In [ ]:

video_stats = {
    'Official Videos': df['is_official_video'].sum(),
    'Lyric Videos': df['is_lyric_video'].sum(),
    'Live Performances': df['is_live_performance'].sum(),
    'Collaborations': df['is_collaboration'].sum()
}

print("="*80)
print("VIDEO METRICS")
print("="*80)
for k, v in video_stats.items():
    print(f"   {k}: {v} ({v/len(df)*100:.1f}%)")


### 7.1. Vistas por Tipo de Video

In [ ]:

df_video = df.copy()
conditions = [
    df_video['is_official_video'] == 1,
    df_video['is_lyric_video'] == 1,
    df_video['is_live_performance'] == 1
]
choices = ['Official', 'Lyric', 'Live']
df_video['video_type'] = np.select(conditions, choices, default='Other')

views_stats = df_video.groupby('video_type').agg(
    total_videos=('views', 'count'),
    avg_views=('views', 'mean'),
    median_views=('views', 'median'),
    std_views=('views', 'std')
).round(2).reset_index()

table_views = views_stats.copy()
table_views['total_videos'] = table_views['total_videos'].astype(int)
table_views['avg_views'] = table_views['avg_views'].apply(lambda x: f"{x:,.0f}")
table_views['median_views'] = table_views['median_views'].apply(lambda x: f"{x:,.0f}")
table_views['std_views'] = table_views['std_views'].apply(lambda x: f"{x:,.0f}")
table_views.columns = ['Video Type', 'Total Videos', 'Avg Views', 'Median Views', 'Std Dev']

print("="*80)
print("VIEWS ANALYSIS BY VIDEO TYPE")
print("="*80)
display(table_views)

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#F9F9F9')
ax.set_facecolor('#F9F9F9')
sns.barplot(data=df_video, x='video_type', y='views', ax=ax, color='#FC4B4C', errorbar='sd')
ax.set_title('Average Views by Video Type', fontweight='bold', fontsize=14)
ax.set_ylabel('Average Views', fontsize=12)
ax.set_xlabel('Video Type', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



### 🤖 Análisis de IA

Los videos de letras lideran con mayor porcentaje de visualizaciones promedio (55%, 12M), indicando una audiencia que valora contenido claro y enfocado en la música.  
El alto compromiso general (12.7%) sugiere una base de seguidores leal, aunque las presentaciones en vivo tienen menor alcance.  
Se recomienda a los artistas priorizar videos de letras y contenido oficial, e integrar elementos en vivo dentro de formatos más accesibles para mantener el interés.


### 7.2. Engagement por Tipo de Video

In [ ]:

engagement_stats = df_video.groupby('video_type').agg(
    total_videos=('engagement', 'count'),
    engagement_rate=('engagement', 'mean'),
    median_engagement=('engagement', 'median'),
    std_engagement=('engagement', 'std')
).round(2).reset_index()

table_engagement = engagement_stats.copy()
table_engagement['total_videos'] = table_engagement['total_videos'].astype(int)
table_engagement['engagement_rate'] = table_engagement['engagement_rate'].round(2).astype(str) + '%'
table_engagement['median_engagement'] = table_engagement['median_engagement'].round(2).astype(str) + '%'
table_engagement['std_engagement'] = table_engagement['std_engagement'].round(2).astype(str) + '%'
table_engagement.columns = ['Video Type', 'Total Videos', 'Avg Engagement', 'Median Engagement', 'Std Dev']

print("="*80)
print("ENGAGEMENT ANALYSIS BY VIDEO TYPE")
print("="*80)
display(table_engagement)

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#F9F9F9')
ax.set_facecolor('#F9F9F9')
sns.barplot(data=df_video, x='video_type', y='engagement', ax=ax, color='#FC4B4C', errorbar='sd')
ax.set_title('Average Engagement by Video Type', fontweight='bold', fontsize=14)
ax.set_ylabel('Engagement (%)', fontsize=12)
ax.set_xlabel('Video Type', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 7.3. Análisis de Duración

In [ ]:

duration_minutes = df['duration_s'] / 60

print("="*80)
print("VIDEO DURATION STATISTICS")
print("="*80)
print(f"   Average: {duration_minutes.mean():.1f} minutes")
print(f"   Minimum: {df['duration_s'].min()} seconds")
print(f"   Maximum: {df['duration_s'].max()} seconds")
print(f"   Median: {df['duration_s'].median()} seconds")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor(YT_BG)

ax1 = axes[0]
ax1.set_facecolor(YT_SURFACE)
n, bins, patches = ax1.hist(duration_minutes, bins=15, edgecolor='white', alpha=0.7, density=True)

for patch in patches:
    patch.set_facecolor('#FE1B1F' if patch.get_height() > 0.2 else '#D8A7A7')

kde = gaussian_kde(duration_minutes)
x = np.linspace(duration_minutes.min(), duration_minutes.max(), 100)
ax1.plot(x, kde(x), color=YT_RED_DARK, linewidth=2.5, label='Density', alpha=0.9)

ax1.axvline(duration_minutes.mean(), color='#220F23', linestyle='--', linewidth=1.5, label=f'Mean: {duration_minutes.mean():.1f} min')
ax1.axvline(duration_minutes.median(), color='#821638', linestyle='--', linewidth=1.5, label=f'Median: {duration_minutes.median():.1f} min')

ax1.set_xlabel('Duration (minutes)', fontsize=10, color=YT_GRAY)
ax1.set_ylabel('Density', fontsize=10, color=YT_GRAY)
ax1.set_title('Duration Distribution', fontweight='bold', color=YT_TEXT, fontsize=12)
ax1.legend(loc='upper right', fontsize=9, facecolor=YT_SURFACE)
ax1.spines[['top', 'right']].set_visible(False)
ax1.grid(True, color=YT_GRID, linestyle='--', alpha=0.5)

ax2 = axes[1]
ax2.set_facecolor(YT_SURFACE)
bp = ax2.boxplot(duration_minutes, vert=False, patch_artist=True, widths=0.6,
                 boxprops=dict(facecolor=YT_RED, color=YT_RED_DARK, alpha=0.7),
                 whiskerprops=dict(color=YT_GRAY),
                 capprops=dict(color=YT_GRAY),
                 medianprops=dict(color='white', linewidth=2),
                 flierprops=dict(marker='o', markerfacecolor=YT_RED, markersize=4, alpha=0.5))
ax2.set_yticks([1])
ax2.set_yticklabels(['Duration'], fontsize=10)
ax2.set_xlabel('Duration (minutes)', fontsize=10, color=YT_GRAY)
ax2.set_title('Key Statistics', fontweight='bold', color=YT_TEXT, fontsize=12)
ax2.spines[['top', 'right']].set_visible(False)
ax2.grid(True, color=YT_GRID, linestyle='--', alpha=0.5, axis='x')

plt.tight_layout()
plt.show()

print(f"\n DURATION STATISTICS:")
print("-"*80)
print(f"   Mean: {duration_minutes.mean():.1f} min | Median: {duration_minutes.median():.1f} min")
print(f"   Min: {duration_minutes.min():.1f} min | Max: {duration_minutes.max():.1f} min")
print(f"   Q1: {duration_minutes.quantile(0.25):.1f} min | Q3: {duration_minutes.quantile(0.75):.1f} min")



### 🤖 Análisis de IA

La duración típica de los videos se concentra estrechamente entre 3 y 4 minutos, como indica la media y mediana similares. La presencia de videos de 0.0 min sugiere contenido fallido o promocional muy breve, mientras que el máximo de 6.9 min muestra flexibilidad para formatos más extensos. Para los creadores, esto refuerza la eficacia del formato corto y estándar, pero sugiere que experimentar con duraciones ligeramente mayores puede ser aceptado sin alejarse significativamente de las expectativas del público.


### 7.4. Distribución por Tipo de Canal

In [ ]:

channel_counts = df['channel_type'].value_counts()

print("\n" + "="*60)
print("CHANNEL TYPE DISTRIBUTION")
print("="*60)

for ch, count in channel_counts.items():
    print(f"   - {ch}: {count} songs ({count/len(df)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(YT_BG)
ax.set_facecolor(YT_SURFACE)

colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(channel_counts)))[::-1]

bars = ax.barh(channel_counts.index, channel_counts.values,
               color=colors, edgecolor='#CC0000', height=0.6, alpha=0.9)

ax.set_xlabel('Number of Songs', fontsize=11, color=YT_GRAY)
ax.set_title('Channel Type Distribution', fontweight='bold', color=YT_TEXT, fontsize=14)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_color(YT_GRID)
ax.xaxis.grid(True, color=YT_GRID, linestyle='--', alpha=0.7)

for bar, val in zip(bars, channel_counts.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val} ({val/len(df)*100:.1f}%)',
            va='center', fontsize=10, fontweight='bold', color=YT_TEXT)

plt.tight_layout()
plt.show()


## 8. Análisis Temporal

### 8.1. Evolución de Vistas por Trimestre

In [ ]:

bg_color = '#F9F9F9'
genre_palette = ['#FF0000', '#282828', '#4A4A4A', '#FFB347', '#FF6B6B']

top5_genres = genre_stats.nlargest(5, 'total_songs')['macro_genre'].tolist()
df_temporal = df[df['macro_genre'].isin(top5_genres)].copy()

temporal_views = df_temporal.groupby(['upload_quarter', 'macro_genre'])['views'].sum().reset_index()
temporal_engagement = df_temporal.groupby(['upload_quarter', 'macro_genre'])['engagement'].mean().reset_index()

fig1, ax1 = plt.subplots(figsize=(12, 6), facecolor=bg_color)
ax1.set_facecolor(bg_color)

sns.lineplot(data=temporal_views, x='upload_quarter', y='views', hue='macro_genre',
             marker='o', palette=genre_palette, linewidth=2.5, ax=ax1)

ax1.set_title('Views Evolution by Quarter (Top 5 Genres)', fontweight='bold', color='#282828', fontsize=14)
ax1.set_xlabel('Release Quarter', color='#282828', fontsize=12)
ax1.set_ylabel('Total Views', color='#282828', fontsize=12)
ax1.tick_params(colors='#282828', labelsize=10)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_color('#4A4A4A')
ax1.spines['bottom'].set_color('#4A4A4A')
ax1.grid(True, linestyle='--', alpha=0.3, color='#AAAAAA')

legend1 = ax1.get_legend()
if legend1:
    legend1.get_frame().set_facecolor(bg_color)
    legend1.get_frame().set_edgecolor('#E5E5E5')
    for text in legend1.get_texts():
        text.set_color('#282828')

plt.tight_layout()
plt.show()


### 8.2. Evolución del Engagement por Trimestre

In [ ]:

fig2, ax2 = plt.subplots(figsize=(12, 6), facecolor=bg_color)
ax2.set_facecolor(bg_color)

sns.lineplot(data=temporal_engagement, x='upload_quarter', y='engagement', hue='macro_genre',
             marker='o', palette=genre_palette, linewidth=2.5, ax=ax2)

ax2.set_title('Engagement Evolution by Quarter (Top 5 Genres)', fontweight='bold', color='#282828', fontsize=14)
ax2.set_xlabel('Release Quarter', color='#282828', fontsize=12)
ax2.set_ylabel('Engagement (%)', color='#282828', fontsize=12)
ax2.tick_params(colors='#282828', labelsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_color('#4A4A4A')
ax2.spines['bottom'].set_color('#4A4A4A')
ax2.grid(True, linestyle='--', alpha=0.3, color='#AAAAAA')

legend2 = ax2.get_legend()
if legend2:
    legend2.get_frame().set_facecolor(bg_color)
    legend2.get_frame().set_edgecolor('#E5E5E5')
    for text in legend2.get_texts():
        text.set_color('#282828')

plt.tight_layout()
plt.show()



### 🤖 Análisis de IA

El análisis revela un patrón estacional marcado, con picos de visualizaciones en el primer y cuarto trimestre, posiblemente vinculado a lanzamientos estratégicos o temporadas festivas. La evolución de la participación muestra un máximo en el tercer trimestre (17.56), a pesar de tener menos vistas, indicando contenido más cautivador en ese período. Una tendencia relevante es la relación inversa observada en el cuarto trimestre: vistas altas con participación baja, sugiriendo que el contenido viral puede no siempre generar una interacción profunda.


### 8.3. Distribución de Lanzamientos por Trimestre

In [ ]:

season_counts = df['upload_quarter'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(YT_BG)
ax.set_facecolor(YT_SURFACE)

bars = ax.bar(season_counts.index, season_counts.values, color='#FC4B4C', edgecolor='white')
ax.set_xlabel('Quarter', fontsize=11, color=YT_GRAY)
ax.set_ylabel('Number of Songs', fontsize=11, color=YT_GRAY)
ax.set_title('Release Distribution by Quarter', fontweight='bold', color=YT_TEXT)
ax.spines[['top', 'right']].set_visible(False)
ax.spines['bottom'].set_color(YT_GRID)
ax.spines['left'].set_color(YT_GRID)
ax.grid(True, color=YT_GRID, linestyle='--', alpha=0.5)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f'{int(height)}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


## 9. Análisis de Colaboraciones

In [ ]:

collab_stats = df.groupby('is_collaboration').agg(
    count=('track_name', 'count'),
    avg_views=('views', 'mean'),
    avg_engagement=('engagement', 'mean')
).reset_index()

collab_stats['is_collaboration'] = collab_stats['is_collaboration'].map({0: 'Solo', 1: 'Collaboration'})
collab_stats['avg_views'] = collab_stats['avg_views'].apply(lambda x: f"{x:,.0f}")
collab_stats['avg_engagement'] = collab_stats['avg_engagement'].round(2).astype(str) + '%'

print("COLLABORATION STATISTICS")
display(collab_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#F9F9F9')
axes[0].set_facecolor('#F9F9F9')
axes[1].set_facecolor('#F9F9F9')

sns.scatterplot(data=df, x='artist_count', y='views', hue='is_collaboration',
                palette={0: 'blue', 1: 'red'}, ax=axes[0], alpha=0.6)
axes[0].set_title('Views vs Number of Artists', fontweight='bold')
axes[0].set_xlabel('Number of Artists')
axes[0].set_ylabel('Views')

sns.scatterplot(data=df, x='artist_count', y='engagement', hue='is_collaboration',
                palette={0: 'blue', 1: 'red'}, ax=axes[1], alpha=0.6)
axes[1].set_title('Engagement vs Number of Artists', fontweight='bold')
axes[1].set_xlabel('Number of Artists')
axes[1].set_ylabel('Engagement (%)')

plt.tight_layout()
plt.show()



### 🤖 Análisis de IA

Las colaboraciones no superan el rendimiento de los temas en solitario en este caso, mostrando un 6% menos de visualizaciones promedio y una tasa de interacción significativamente menor (6.2% vs. 13.7%). Esto podría deberse a una dilución de la identidad artística o a que el público principal valora más el trabajo individual del artista. Se recomienda ser selectivo con las colaboraciones, priorizando aquellas que aporten un valor artístico claro y se alineen con la base de seguidores existente para no comprometer el compromiso (*engagement*).


## 10. Resumen Ejecutivo


### ### 🤖 Análisis de IA

**Resumen Ejecutivo del Análisis de Charts Musicales**

El análisis global de 100 canciones de 30 países revela un panorama diverso con 1.150 millones de visualizaciones y 134 millones de "me gusta". Geográficamente, India lidera en volumen de canciones (23), mientras que Estados Unidos domina en interacción, generando 42,3 millones de likes. Esto indica un mercado indio masivo en producción, pero uno estadounidense con mayor capacidad de generar compromiso por canción. En géneros, el R&B/Soul muestra una tasa de engagement excepcional (160,4%), muy superior al promedio del 12,7%, señalándolo como un estilo altamente conectivo. El formato de video "Lyric" es el más efectivo, y las canciones en solitario superan en un 7% las vistas a las colaboraciones, sugiriendo un posible cansancio del mercado hacia proyectos conjuntos.

**Tendencias y Recomendaciones Estratégicas:**
1.  **Enfoque Regional:** Priorizar promoción en EE.UU. para maximizar engagement, e India para volumen de lanzamientos.
2.  **Estrategia de Género:** Invertir en producción de R&B/Soul y géneros alternativos, que muestran comunidades digitales muy comprometidas.
3.  **Contenido Optimo:** Utilizar videos de tipo "Lyric" como formato principal para capitalizar su alta efectividad demostrada.
4.  **Modelo Artístico:** Reevaluar la saturación de colaboraciones; potenciar lanzamientos en solitario que, según los datos, capturan mayor atención inicial.

La conclusión principal es la desconexión entre volumen de producción y engagement efectivo, requiriendo estrategias diferenciadas por mercado y un contenido altamente adaptado al formato digital de consumo actual.


---
**Análisis Completado** | Generado por Music Charts Intelligence | Insights de IA por DeepSeek